# 02 — Segmentation Analysis
**Auction Marketplace Segmentation & Price Intelligence Engine**

This notebook walks through the K-Means segmentation process, cluster validation, and business interpretation of each discovered segment.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import json
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid')
print('Ready')

## 1. Load Feature-Engineered Data

In [ ]:
df = pd.read_csv('../data/processed/listings_features.csv', parse_dates=['list_date'])
print(f"Shape: {df.shape}")
print(f"Segment distribution:")
if 'segment_label' in df.columns:
    print(df['segment_label'].value_counts())

## 2. Segmentation Feature Set

In [ ]:
SEG_FEATURES = [
    'log_asking_price', 'price_vs_cat_median', 'price_vs_cond_median',
    'age_years', 'log_hours_used', 'hours_vs_cat_median',
    'total_engagement', 'bid_rate', 'inquiry_rate',
    'condition_score', 'seller_listing_count', 'seller_sold_rate',
    'category_freq', 'region_demand_proxy', 'log_views',
]

X = df[SEG_FEATURES].fillna(df[SEG_FEATURES].median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Feature matrix: {X_scaled.shape}")

## 3. Elbow & Silhouette Method

In [ ]:
# Load pre-computed k-selection results
k_path = '../data/processed/k_selection.json'
if os.path.exists(k_path):
    with open(k_path) as f:
        k_results = json.load(f)

    ks = [int(k) for k in k_results.keys()]
    sils = [k_results[str(k)]['silhouette'] for k in ks]
    inertias = [k_results[str(k)]['inertia'] for k in ks]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.plot(ks, inertias, 'bo-', linewidth=2, markersize=8)
    ax1.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='Selected k=5')
    ax1.set_xlabel('k'); ax1.set_ylabel('Inertia'); ax1.set_title('Elbow Method'); ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(ks, sils, 'rs-', linewidth=2, markersize=8)
    ax2.axvline(x=5, color='blue', linestyle='--', alpha=0.5, label='Selected k=5')
    ax2.set_xlabel('k'); ax2.set_ylabel('Silhouette Score'); ax2.set_title('Silhouette Score vs k'); ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('../reports/figures/elbow_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Best silhouette at k={ks[sils.index(max(sils))]}: {max(sils):.4f}")
else:
    print("Run train_segmentation.py first to generate k_selection.json")

## 4. Segment Profiles

In [ ]:
if 'segment_label' in df.columns:
    profile_cols = ['asking_price','age_years','hours_used','total_engagement',
                    'bid_rate','condition_score','seller_sold_rate','days_on_market','sold']

    profile = df.groupby('segment_label')[profile_cols].agg(['mean','median']).round(2)
    profile.columns = ['_'.join(c) for c in profile.columns]
    profile['count'] = df.groupby('segment_label')['listing_id'].count()
    print(profile[['count','asking_price_mean','age_years_mean','total_engagement_mean','sold_mean']].to_string())

## 5. PCA Cluster Visualization

In [ ]:
if 'segment_kmeans' in df.columns:
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X_scaled)
    var_exp = pca.explained_variance_ratio_

    SEGMENT_NAMES = {
        0: 'Premium High-Demand',
        1: 'Aging Low-Engagement',
        2: 'Value Fleet Inventory',
        3: 'Hot Fast-Moving Deals',
        4: 'Niche Specialty Equipment',
    }

    fig, ax = plt.subplots(figsize=(11, 7))
    colors = plt.cm.tab10(np.linspace(0, 0.9, 5))

    for i in range(5):
        mask = df['segment_kmeans'] == i
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=[colors[i]], alpha=0.4, s=8,
                   label=SEGMENT_NAMES.get(i, f'Seg {i}'))

    ax.set_xlabel(f'PC1 ({var_exp[0]:.1%} variance explained)')
    ax.set_ylabel(f'PC2 ({var_exp[1]:.1%} variance explained)')
    ax.set_title('K-Means Segments — PCA Projection')
    ax.legend(markerscale=4, fontsize=10)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig('../reports/figures/pca_clusters_notebook.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Price Distribution by Segment

In [ ]:
if 'segment_label' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = plt.cm.Set2(np.linspace(0, 1, 5))

    for i, (seg, group) in enumerate(df.groupby('segment_label')):
        prices = group['asking_price'].clip(upper=group['asking_price'].quantile(0.99))
        prices.plot.kde(ax=ax, label=seg, color=colors[i], linewidth=2)

    ax.set_xlabel('Asking Price ($)')
    ax.set_title('Price Density by Segment')
    ax.set_xlim(0, df['asking_price'].quantile(0.98))
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('../reports/figures/segment_price_density.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Business Interpretation

| Segment | Name | Characteristics | Strategy |
|---|---|---|---|
| 0 | Premium High-Demand | High price, good condition, high engagement | Premium listing, hold price |
| 1 | Aging Low-Engagement | Old equipment, low engagement, low price | Discount or remarket |
| 2 | Value Fleet Inventory | Mid-price, high volume sellers, fair condition | Bundle deals, fleet pricing |
| 3 | Hot Fast-Moving Deals | Moderate price, very high engagement, fast sale | Quick re-list, competitive pricing |
| 4 | Niche Specialty Equipment | High price, low volume category, specific demand | Target specialist buyers |

In [ ]:
print('Segmentation analysis complete.')